# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, every entity (record set, field, column) has a unique `@id`. Here we list all the record sets and their fields, referencing each by its `@id`.

In [ ]:
# List all record sets and their fields using @id
recordset_list = []
if hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    Field @id: {field['@id']} | name: {field.get('name','N/A')} | dataType: {field.get('dataType','N/A')}")
        else:
            print("  No fields listed.")
        recordset_list.append(rs['@id'])
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

Below, we load all available record sets, referencing each by its `@id` in Croissant.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

# Use recordset_list from previous step
for record_set_id in recordset_list:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id} | Columns: {df.columns.tolist()}")

# For demonstration, pick the first available record set
if recordset_list:
    display_id = recordset_list[0]
    print(f"Displaying top records of RecordSet @id {display_id}:")
    display(dataframes[display_id].head())
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming distributions, or grouping data by key attributes. All entities are referenced by their Croissant `@id`.

**Let's select a numeric field (@id) from our record set for illustration.**

In [ ]:
# Example EDA on a record set
from IPython.display import display

# Choose a record set and a numeric field @id
if recordset_list:
    record_set_id = recordset_list[0]
    df = dataframes[record_set_id]
    # Find a candidate numeric field (e.g., age, hormone levels etc.)
    numeric_fields = [col for col in df.columns if 'age' in col.lower() or df[col].dtype in ['float64', 'int64']]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field @id: '{numeric_field}' for EDA.")

        # Filtering
        threshold = 30  # Example threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalization
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Grouping by another field (e.g., 'infertility', 'phenotype', etc.)
        group_fields = [col for col in df.columns if 'phenotype' in col.lower() or 'infertility' in col.lower()]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by {group_field}:")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in record set.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric field, and a scatter plot of two fields (where available) using Matplotlib and Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field from the previous section
if recordset_list and numeric_fields:
    record_set_id = recordset_list[0]
    df = dataframes[record_set_id]
    numeric_field = numeric_fields[0]

    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.show()

    # If there is a second numeric field, plot scatter
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f"Scatter: {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numeric fields for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the Croissant `@id` references allowed precise, reproducible access to all entities (record sets, fields, columns).
- Dataset provides tabulated clinical, hormone, imaging, and genetic variant information for three female patients with non-classical 21-hydroxylase deficiency and infertility, supporting rare disease research.
- Sample EDA illustrated filtering, normalization, and grouping on numeric clinical features, as well as basic visualization.
- Limitations include small cohort size, gender and geographic homogeneity, and the dataset is recommended for academic illustration rather than predictive modeling.
- For additional use, all fields and record set `@id`s (see the overview) can be referenced programmatically for further reproducibility.